# Routes & Stops Selection
## Goal
- Prepare the following datasets:
    1. List of unique routes
    2. Selected bus stops for each route
    3. Target encoding for each adjacent bus stop pair for each route

## (for future organization)
## Route Selection
1. Use `data/bus_routes_mar28.csv` as the ground truth
2. After getting `final_training_data.parquet`, export all unique routes (because these are the only routes that can be predicted)

## Stops Selection
1. Create a `stops.json` with {"StopID": "RouteName_Zh"}
1. From `bus_event_time.parquet`, get the number of trips recorded per route per stop {"StopID": number of trips}
    - When counting the number of trips recorded per route per stop, use max("進站","出站")
2. Sort the dictonary by values
4. for routes with few stops (< 10) 
    1. go through all the routes from most recorded to least
    2. if the current stop is within 4 of any existing stops in our collection, abandon the stop
    3. if the current stop is at least 4 km away from any existing stops in our collection, append it in the collection
5. for routes with more stops (> 10)
    1. go through only half of all the stops (from most recorded to least)
    2. same algorithm as above
6. Create `target_stops.json` ({"route": list[StopID]})
    - Sort list[StopID] by StopSeq in that route

# Import

In [ ]:
import json
import polars as pl
from constants import DATA_FILE, DATA_TEST_FILE, PROCESSED_DATA_FOLDER, STOPS
from helpers import calculate_distance_meter, clean_df

pl.Config.set_tbl_rows(30)

polars.config.Config

# Routes
- Create `target_routes.json` containing the routes that will be covered in final model  training

In [2]:
unique_routes_train = (
    pl.scan_parquet(DATA_FILE).select(pl.col("SubRouteID").unique()).collect()
)
unique_routes_test = (
    pl.scan_parquet(DATA_TEST_FILE).select(pl.col("SubRouteID").unique()).collect()
)

In [ ]:
train_set = set(unique_routes_train["SubRouteID"].to_list())
test_set = set(unique_routes_test["SubRouteID"].to_list())

intersection = sorted(list(train_set & test_set))

with open(PROCESSED_DATA_FOLDER / "target_routes.json", "w") as f:
    json.dump(intersection, f)

print(f"Train routes: {len(train_set)}")
print(f"Test routes: {len(test_set)}")
print(f"Intersection: {len(intersection)}")

Train routes: 1801
Test routes: 1668
Intersection: 1667


# Stops
- Create a dictionary mapping from `StopID` to `StopName` for easier reference later

In [ ]:
df_stops = pl.read_csv(STOPS)
stops_dict = dict(
    zip(df_stops["StopID"].to_list(), df_stops["StopNameZh_tw"].to_list())
)
with open(PROCESSED_DATA_FOLDER / "stops_dict.json", "w", encoding="utf-8") as f:
    json.dump(stops_dict, f, ensure_ascii=False)

# Selected Stops per Route

In [ ]:
with open(PROCESSED_DATA_FOLDER / "target_routes.json", "r") as f:
    routes = json.load(f)
df_stops = pl.read_csv(STOPS)
df_train = pl.scan_parquet(DATA_FILE).filter(pl.col("SubRouteID").is_in(routes))
df = df_train.pipe(clean_df)

In [ ]:
# ── Count trips recorded per Route per StopID ──────────────────────────────
trips = (
    df.filter(pl.col("Event").is_in(["進站", "離站"]))
    .group_by(["Route", "StopID", "Event"])
    .agg(pl.col("Time").len().alias("count"))
    .group_by(["Route", "StopID"])
    .agg(pl.col("count").max().alias("Trips_recorded"))
    .collect(engine="streaming")
)

# ── Build stop coordinate dictionary for faster lookup {StopID -> (lat, lon)} ────────────────────
stop_coords: dict[int, tuple[float, float]] = {
    row["StopID"]: (row["PositionLat"], row["PositionLon"])
    for row in df_stops.select(["StopID", "PositionLat", "PositionLon"]).to_dicts()
}

# ── Build StopID -> StopSequence lookup per Route ──────────────────────────
seq_lookup: dict[tuple[str, int], int] = {
    (row["Route"], row["StopID"]): row["StopSeq"]
    for row in (
        df.select(["Route", "StopID", "StopSeq"])
        .unique(subset=["Route", "StopID"])
        .collect(engine="streaming")
        .to_dicts()
    )
}

# ── Determine number of unique stops per route ─────────────────────────────
stop_counts = trips.group_by("Route").agg(pl.col("StopID").n_unique().alias("n_stops"))
few_stops_routes = set(stop_counts.filter(pl.col("n_stops") < 10)["Route"].to_list())

# ── Selection algorithm ────────────────────────────────────────────────────
DISTANCE_THRESHOLD_KM = 4.0

target_stops: dict[str, list[int]] = {}

for route, group in trips.group_by("Route"):
    route: str = route[0]

    # Sort candidates by Trips_recorded descending
    candidates = group.sort("Trips_recorded", descending=True)["StopID"].to_list()

    # For routes with many stops, only consider the top half.
    if route not in few_stops_routes:
        half = max(1, len(candidates) // 2)
        candidates = candidates[:half]

    collected: list[int] = []  # selected stops for this route

    for stop_id in candidates:
        if stop_id not in stop_coords:
            continue  # skip stops with no coordinate data
        lat, lon = stop_coords[stop_id]

        too_close = False
        for existing_id in collected:
            if existing_id not in stop_coords:
                continue
            e_lat, e_lon = stop_coords[existing_id]
            if (
                calculate_distance_meter(lat, lon, e_lat, e_lon) / 1000
                < DISTANCE_THRESHOLD_KM
            ):
                too_close = True
                break

        if not too_close:
            collected.append(stop_id)

    # Sort collected stops by StopSequence in this route.
    collected.sort(key=lambda sid: seq_lookup.get((route, sid), 0))
    target_stops[route] = collected

# ── Save output ───────────────────────────────────────────────────────────
with open(PROCESSED_DATA_FOLDER / "target_stops.json", "w", encoding="utf-8") as f:
    json.dump(target_stops, f, ensure_ascii=False, indent=2)

print(f"Done. {len(target_stops)} routes written to target_stops.json.")

Done. 1666 routes written to target_stops.json.


In [3]:
with open(PROCESSED_DATA_FOLDER / "target_stops.json", "r") as f:
    target_stops = json.load(f)
with open(PROCESSED_DATA_FOLDER / "stops.json", "r", encoding="utf-8") as f:
    stops_dict = json.load(f)

In [4]:
df_target_stops = pl.DataFrame(
    [{"Route": k, "n_Stops": len(v)} for k, v in target_stops.items()]
)

## Check output

In [5]:
df_target_stops.filter(pl.col("n_Stops") == 1).sort("Route")

Route,n_Stops
str,i64
"""0968A1""",1
"""103201""",1
"""1558A2""",1
"""161101""",1
"""181802""",1
"""1838A2""",1
"""187202""",1
"""201102""",1
"""2088A1""",1


In [9]:
check = "0968A1"
for s in target_stops[check]:
    print(s, stops_dict.get(str(s)))

300174 開南大學


In [12]:
df.filter(pl.col("Route") == check).collect().group_by(["Stop", "Event"]).agg(
    pl.col("Time").n_unique().alias("count")
).group_by("Stop").agg(pl.col("count").max().alias("Trips_recorded")).sort(
    "Trips_recorded", descending=True
)

Stop,Trips_recorded
str,u32
"""開南大學""",496
"""大竹消防隊""",491
"""大竹""",489
"""南竹蘆興街口""",486
"""大竹上興路口""",485
"""上大竹""",483
"""南竹路四段37號""",483
"""光明國小""",481
"""庫倫街口""",480


- For this route, the algorithm is supposed to pick up 庫倫街口 which is a major transport hub in Taipei's side of the route. The algorithm was supposed to weed out stops that are too close or stops with too few records, but for a balanced, well-reeocrded routes like this, it filters out desired stops.
- To address this issue, abandon the step (in the algorithm) that searches only half of the stops if there are more than 10 stops on that route. This should allow the algorithm to pick 庫倫街口 for the route.

In [15]:
check = "666901"
for s in target_stops[check]:
    print(s, stops_dict.get(str(s)))

247989 孔雀園


- 666901 is an extremely short route that surrounds the Sun Moon Lake. In my opinion, there's not much value in predicting extremely short routes like this. Therefore, we can abandon routes like this in the model training.

# Updated Approach 

In [ ]:
# ── Count trips recorded per Route per StopID ──────────────────────────────
trips = (
    df.filter(pl.col("Event").is_in(["進站", "離站"]))
    .group_by(["Route", "StopID", "Event"])
    .agg(pl.col("Time").len().alias("count"))
    .group_by(["Route", "StopID"])
    .agg(pl.col("count").max().alias("Trips_recorded"))
    .sort("Trips_recorded")
    .collect(engine="streaming")
)

# ── Build stop coordinate dictionary for faster lookup {StopID -> (lat, lon)} ────────────────────
stop_coords: dict[int, tuple[float, float]] = {
    row["StopID"]: (row["PositionLat"], row["PositionLon"])
    for row in df_stops.select(["StopID", "PositionLat", "PositionLon"]).to_dicts()
}

# ── Build StopID -> StopSequence lookup per Route ──────────────────────────
seq_lookup: dict[tuple[str, int], int] = {
    (row["Route"], row["StopID"]): row["StopSeq"]
    for row in (
        df.select(["Route", "StopID", "StopSeq"])
        .unique(subset=["Route", "StopID"])
        .collect(engine="streaming")
        .to_dicts()
    )
}

# ── Selection algorithm ────────────────────────────────────────────────────
DISTANCE_THRESHOLD_KM = 4.0

target_stops: dict[str, list[int]] = {}

for route, group in trips.group_by("Route"):
    route: str = route[0]

    # Sort candidates by Trips_recorded descending
    candidates = group.sort("Trips_recorded", descending=True)["StopID"].to_list()

    collected: list[int] = []  # selected stops for this route

    # O(M*N) potential bottleneck if there are even more routes
    for stop_id in candidates:
        if stop_id not in stop_coords:
            continue  # skip stops with no coordinate data
        lat, lon = stop_coords[stop_id]

        too_close = False
        for existing_id in collected:
            if existing_id not in stop_coords:
                continue
            e_lat, e_lon = stop_coords[existing_id]
            if (
                calculate_distance_meter(lat, lon, e_lat, e_lon) / 1000
                < DISTANCE_THRESHOLD_KM
            ):
                too_close = True
                break

        if not too_close:
            collected.append(stop_id)

    # Sort collected stops by StopSequence in this route.
    collected.sort(key=lambda sid: seq_lookup.get((route, sid), 0))
    target_stops[route] = collected

# ── Save output ───────────────────────────────────────────────────────────
with open(PROCESSED_DATA_FOLDER / "target_stops.json", "w", encoding="utf-8") as f:
    json.dump(target_stops, f, ensure_ascii=False, indent=2)

print(f"Done. {len(target_stops)} routes written to target_stops.json.")


Done. 1666 routes written to target_stops.json.


## Check output

In [ ]:
with open(PROCESSED_DATA_FOLDER / "target_stops.json", "r") as f:
    target_stops = json.load(f)
with open(PROCESSED_DATA_FOLDER / "stops_dict.json", "r", encoding="utf-8") as f:
    stops_dict = json.load(f)
df_target_stops = pl.DataFrame(
    [{"Route": k, "n_Stops": len(v)} for k, v in target_stops.items()]
)

In [7]:
df_target_stops.filter(pl.col("n_Stops") == 1).sort("Route")

Route,n_Stops
str,i64
"""161101""",1
"""627701""",1
"""627702""",1
"""666901""",1
"""666902""",1
"""6669A1""",1
"""6669A2""",1
"""812802""",1
"""815701""",1


- Routes with 1 stop selected dropped from 27 to 9.

In [17]:
check = "161101"
for s in target_stops[check]:
    print(s, stops_dict[str(s)])
df.filter(pl.col("Route") == "161101").collect()

289792 臺北轉運站


Route,Plate,Direction,Stop,StopID,StopSeq,Event,Time,Day_of_week
str,str,cat,str,i32,i32,cat,datetime[μs],cat
"""161101""","""066-FZ""","""去程""","""臺北轉運站""",289792,1,"""進站""",2025-12-03 16:47:41,"""Wed"""
"""161101""","""066-FZ""","""去程""","""臺北轉運站""",289792,1,"""離站""",2025-12-03 16:49:58,"""Wed"""
"""161101""","""709-U5""","""去程""","""臺北轉運站""",289792,1,"""進站""",2025-12-17 16:51:41,"""Wed"""
"""161101""","""709-U5""","""去程""","""臺北轉運站""",289792,1,"""離站""",2025-12-17 16:53:44,"""Wed"""


- All the routes left are extremely short routes. They are safe to be abandoned
- The exception is 161101, which after investigation is confirmed to be data source issue, meaning it should be abandoned as well.

# Update `target_routes.json`

In [ ]:
with open(PROCESSED_DATA_FOLDER / "target_routes.json", "r", encoding="utf-8") as f:
    routes = json.load(f)
abandoned_routes = df_target_stops.filter(pl.col("n_Stops") == 1)["Route"].to_list()
new_routes = [r for r in routes if r not in abandoned_routes]
with open(PROCESSED_DATA_FOLDER / "target_routes.json", "w", encoding="utf-8") as f:
    json.dump(new_routes, f)

# Summary
In this notebook, I
1. finalized the list of all routes in the final model
2. finalized the select list of stops that will be incorporated in final model for each route